# INF2006 — Task 1: Build the Raw Data Corpus
**Sources:** Daily Price CSV (required) + GDELT (optional supporting source)  
**Modes:** `local` for development, `emr` for final HDFS staging and verification.

> **Important:** This notebook is a starting implementation, not evidence that Task 1 is complete. Replace configuration placeholders with your actual files and source references. Keep all original files unchanged. Task 2 performs cleaning. The assessment requires HDFS on AWS EMR and a fresh-session rerun. No raw datasets should be included in the submission ZIP.

**Reference scope from project brief (not guaranteed to match your downloaded files):** prices: 505 series, 605,887 accepted bars, 2013-02-08 to 2017-12-29; GDELT: 731 daily archives, July 2015–June 2017, approximately 9.61 GB compressed / 54.71 GB expanded. Record *measured* counts and coverage, not assumed figures.

## 1. Configuration and prerequisites
**Where:** local Jupyter on Mac initially; later rerun on an EMR-connected notebook.

Set `PRICE_FILE`, `GDELT_DIR`, and `SOURCE_REFERENCES`. If files are stored elsewhere on EMR, update these paths there. `RUN_MODE='local'` avoids any AWS charges. Do not set `RUN_MODE='emr'` until the required EMR environment is ready.

Python standard library and pandas are used for **local exploratory inspection only**. The final assessed processing uses Hadoop/HDFS/Hive/PySpark on EMR. GDELT inspection below streams ZIP members instead of loading the full corpus into memory.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, csv, io, json, hashlib, zipfile, subprocess, re
from collections import Counter
import pandas as pd

# Running locally on your Mac
RUN_MODE = "local"

# Main project folder
PROJECT_DIR = Path.cwd()

# Actual location of your stock dataset
PRICE_FILE = PROJECT_DIR / "dataset" / "all_stocks_5yr.csv"

# Actual location of your GDELT dataset
GDELT_DIR = PROJECT_DIR / "dataset" / "gdelt"

# Folder to store Task 1 outputs
OUTPUT_DIR = PROJECT_DIR / "task1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset source information
SOURCE_REFERENCES = {
    "prices": "REPLACE_WITH_LECTURER_SOURCE_OR_DOWNLOAD_REFERENCE",
    "gdelt": "REPLACE_WITH_EXACT_GDELT_SOURCE_ARCHIVE_REFERENCE"
}

ACQUIRED_UTC = "REPLACE_WITH_ACTUAL_DOWNLOAD_DATE_UTC"

# HDFS location for later EMR execution
HDFS_BASE = "/project/raw"

# Verify file locations
print("Mode:", RUN_MODE)
print("Project directory:", PROJECT_DIR)
print("Price file exists:", PRICE_FILE.exists(), PRICE_FILE)
print("GDELT folder exists:", GDELT_DIR.exists(), GDELT_DIR)

Mode: local
Project directory: /Users/wileen/Desktop/y2sit/cloud/bigdataAssignment
Price file exists: True /Users/wileen/Desktop/y2sit/cloud/bigdataAssignment/dataset/all_stocks_5yr.csv
GDELT folder exists: False /Users/wileen/Desktop/y2sit/cloud/bigdataAssignment/dataset/gdelt


## 2. Raw-file inventory and provenance
**Goal:** identify exact files, sizes, SHA-256 checksums, acquisition details, and source versions. Checksums enable exact-file verification. This step does not modify the originals.

**Note:** hashing every large GDELT archive takes time; the result is a genuine per-file inventory, not a guess.

In [2]:
def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

price_files = [PRICE_FILE] if PRICE_FILE.is_file() else []
gdelt_files = sorted(p for p in GDELT_DIR.rglob("*") if p.is_file()) if GDELT_DIR.is_dir() else []
print("Price files:", len(price_files), "GDELT files:", len(gdelt_files))
print("GDELT file extensions:", dict(Counter(p.suffix.lower() for p in gdelt_files)))

inventory = []
for source, paths in [("prices", price_files), ("gdelt", gdelt_files)]:
    for p in paths:
        inventory.append({
            "source": source, "filename": p.name,
            "relative_path": str(p.relative_to(PROJECT_DIR)) if p.is_relative_to(PROJECT_DIR) else str(p),
            "bytes": p.stat().st_size, "sha256": sha256_file(p),
            "acquired_utc": ACQUIRED_UTC,
            "source_reference": SOURCE_REFERENCES[source]
        })
inventory_df = pd.DataFrame(inventory)
display(inventory_df.head(10))
inventory_df.to_csv(OUTPUT_DIR / "raw_file_inventory.csv", index=False)
print("Total measured bytes:", inventory_df['bytes'].sum() if len(inventory_df) else 0)

Price files: 1 GDELT files: 0
GDELT file extensions: {}


,source,filename,relative_path,bytes,sha256,acquired_utc,source_reference
0,prices,all_stocks_5yr.csv,dataset/all_stocks_5yr.csv,29580549,6aea253cd19de60b568143991aaf1fa482456565c38920...,REPLACE_WITH_ACTUAL_DOWNLOAD_DATE_UTC,REPLACE_WITH_LECTURER_SOURCE_OR_DOWNLOAD_REFER...


Total measured bytes: 29580549


## 3. Inspect the price CSV (small preview)
**Goal:** discover the *actual* schema before writing validation rules. The provided PDF does not define the exact column names, identifier format, currency, or time zone. Do not invent them.

Only a sample is loaded initially; the full-file quality audit below reads the CSV in chunks.

In [3]:
if PRICE_FILE.exists():
    price_preview = pd.read_csv(PRICE_FILE, nrows=10, low_memory=False)
    display(price_preview)
    print("Columns:", list(price_preview.columns))
    print("Inferred sample types:", price_preview.dtypes.astype(str).to_dict())
else:
    print("Price CSV not found: update PRICE_FILE in Section 1.")

,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL
5,2013-02-15,13.93,14.61,13.93,14.50,15628000,AAL
6,2013-02-19,14.33,14.56,14.08,14.26,11354400,AAL
7,2013-02-20,14.17,14.26,13.15,13.33,14725200,AAL
8,2013-02-21,13.62,13.95,12.90,13.37,11922100,AAL
9,2013-02-22,13.57,13.60,13.21,13.57,6071400,AAL


Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'Name']
Inferred sample types: {'date': 'str', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'int64', 'Name': 'str'}


## 4. Map price fields and audit data quality
**Before running:** edit the names below to match the actual columns printed in Section 3. The audit checks exact duplicates, repeated security/date keys, missing values, date parsing, numeric validity, and OHLC consistency. It **does not delete or repair** raw records.

Blank strings are counted as missing. The full audit reads chunks, so it is practical for the reference price CSV. `seen_keys` is used only for the modest price corpus, not GDELT.

In [5]:
PRICE_COLUMNS = {
    "security": "Name",
    "date": "date",
    "open": "open",
    "high": "high",
    "low": "low",
    "close": "close",
    "volume": "volume"
}

DATE_FORMAT = "%Y-%m-%d"

def audit_prices(path, cols, chunk_size=50_000):
    if not path.exists():
        raise FileNotFoundError(path)
    actual = pd.read_csv(path, nrows=0).columns.tolist()
    missing_columns = [v for v in cols.values() if v not in actual]
    if missing_columns:
        raise ValueError(f"Update PRICE_COLUMNS first. Missing: {missing_columns}; actual: {actual}")
    totals = Counter(); missing = Counter(); seen_keys = set(); seen_rows = set()
    first_date = None; last_date = None; identifiers = set()
    for chunk in pd.read_csv(path, chunksize=chunk_size, dtype=str, keep_default_na=False):
        totals['rows'] += len(chunk)
        blank = chunk.apply(lambda s: s.astype(str).str.strip().eq(''))
        missing.update(blank.sum().to_dict())
        d = pd.to_datetime(chunk[cols['date']].replace('', pd.NA), format=DATE_FORMAT, errors='coerce')
        totals['invalid_dates'] += int(d.isna().sum())
        if d.notna().any():
            low, high = d.min(), d.max()
            first_date = low if first_date is None else min(first_date, low)
            last_date = high if last_date is None else max(last_date, high)
        identifiers.update(chunk[cols['security']].dropna().unique())
        numeric = {k: pd.to_numeric(chunk[v].replace('', pd.NA), errors='coerce') for k,v in cols.items() if k in ('open','high','low','close','volume')}
        for k, s in numeric.items():
            totals[f'invalid_or_missing_{k}'] += int(s.isna().sum())
            totals[f'nonpositive_{k}' if k != 'volume' else 'negative_volume'] += int((s <= 0).sum() if k != 'volume' else (s < 0).sum())
        o,h,l,c = (numeric[k] for k in ('open','high','low','close'))
        totals['high_below_low'] += int((h < l).sum())
        totals['open_outside_range'] += int(((o < l) | (o > h)).sum())
        totals['close_outside_range'] += int(((c < l) | (c > h)).sum())
        for row in chunk.itertuples(index=False, name=None):
            if row in seen_rows: totals['exact_duplicate_rows'] += 1
            else: seen_rows.add(row)
        for key in zip(chunk[cols['security']], chunk[cols['date']]):
            if key in seen_keys: totals['duplicate_security_date'] += 1
            else: seen_keys.add(key)
    return {**dict(totals), 'distinct_security_ids':len(identifiers),
            'earliest_date':str(first_date.date()) if first_date is not None else None,
            'latest_date':str(last_date.date()) if last_date is not None else None,
            'missing_by_column':dict(missing), 'schema_columns':actual}

price_quality = {}
if PRICE_FILE.exists() and all(v in pd.read_csv(PRICE_FILE, nrows=0).columns for v in PRICE_COLUMNS.values()):
    price_quality = audit_prices(PRICE_FILE, PRICE_COLUMNS)
    print(json.dumps(price_quality, indent=2))
else:
    print('ACTION: Set PRICE_COLUMNS to the actual names, then rerun this cell.')

{
  "rows": 619040,
  "invalid_dates": 0,
  "invalid_or_missing_open": 11,
  "nonpositive_open": 0,
  "invalid_or_missing_high": 8,
  "nonpositive_high": 0,
  "invalid_or_missing_low": 8,
  "nonpositive_low": 0,
  "invalid_or_missing_close": 0,
  "nonpositive_close": 0,
  "invalid_or_missing_volume": 0,
  "negative_volume": 0,
  "high_below_low": 1,
  "open_outside_range": 4,
  "close_outside_range": 9,
  "distinct_security_ids": 505,
  "earliest_date": "2013-02-08",
  "latest_date": "2018-02-07",
  "missing_by_column": {
    "date": 0,
    "open": 11,
    "high": 8,
    "low": 8,
    "close": 0,
    "volume": 0,
    "Name": 0
  },
  "schema_columns": [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "Name"
  ]
}


## 5. Inspect GDELT archives without extracting everything
**Goal:** record the archive types, ZIP members, sample record structure, and a limited preview. The brief does not state whether your files are GDELT event exports, GKG, or another archive format. Inspect first.

This section **samples** archives only; sampled results must not be described as complete GDELT corpus statistics. Full GDELT row counts/quality checks must be computed in the assessed distributed processing (Tasks 1–2) or through a separate full streaming audit.

In [8]:
def inspect_gdelt_archive(path, max_lines=5):
    result = {
        'filename': path.name,
        'type': path.suffix.lower(),
        'members': [],
        'sample_lines': []
    }

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            result['members'] = [
                {
                    'name': x.filename,
                    'uncompressed_bytes': x.file_size,
                    'compressed_bytes': x.compress_size
                }
                for x in z.infolist()
            ]

            for member in z.infolist():
                if member.is_dir():
                    continue

                with z.open(member) as raw:
                    wrapper = io.TextIOWrapper(
                        raw,
                        encoding='utf-8',
                        errors='replace'
                    )

                    for _, line in zip(range(max_lines), wrapper):
                        result['sample_lines'].append(
                            line.rstrip('\n')[:500]
                        )
                break

    elif path.suffix.lower() in {'.csv', '.tsv', '.txt'}:
        with open(path, encoding='utf-8', errors='replace') as f:
            result['sample_lines'] = [
                line.rstrip('\n')[:500]
                for _, line in zip(range(max_lines), f)
            ]

    else:
        result['note'] = 'Unknown format: inspect separately; no schema assumptions made.'

    return result


# Find all GDELT ZIP files
gdelt_files = sorted(GDELT_DIR.glob("*.zip"))

# Inspect all 3 files
sample_gdelt = [
    inspect_gdelt_archive(p)
    for p in gdelt_files
]

# Display results
for item in sample_gdelt:
    print(json.dumps(item, indent=2)[:5000])

if not gdelt_files:
    print('ACTION: Update GDELT_DIR in Section 1.')
else:
    print("\nGDELT inspection done")
    print("Total files inspected:", len(sample_gdelt))

{
  "filename": "20171227.export.CSV.zip",
  "type": ".zip",
  "members": [
    {
      "name": "20171227.export.CSV",
      "uncompressed_bytes": 51757873,
      "compressed_bytes": 8918681
    }
  ],
  "sample_lines": [
    "718045508\t20071230\t200712\t2007\t2007.9863\tPOL\tPOLAND\tPOL\t\t\t\t\t\t\t\tDNK\tDENMARK\tDNK\t\t\t\t\t\t\t\t0\t070\t070\t07\t2\t7.0\t2\t1\t2\t7.75193798449612\t1\tDenmark\tDA\tDA\t56\t10\tDA\t3\tTown Of Denmark, Maine, United States\tUS\tUSME\t43.979\t-70.8064\t582438\t1\tPoland\tPL\tPL\t52\t20\tPL\t20171227\thttps://www.conwaydailysun.com/berlin_sun/business/local/poland-spring-gives-library-grant-in-denmark/article_4d427e68-e736-11e7-aaf0-8b45bac337a1.html",
    "718045509\t20071230\t200712\t2007\t2007.9863\tPOL\tPOLAND\tPOL\t\t\t\t\t\t\t\tDNK\tDENMARK\tDNK\t\t\t\t\t\t\t\t0\t070\t070\t07\t2\t7.0\t4\t1\t4\t7.75193798449612\t1\tPoland\tPL\tPL\t52\t20\tPL\t3\tTown Of Denmark, Maine, United States\tUS\tUSME\t43.979\t-70.8064\t582438\t1\tPoland\tPL\tPL\t52\t20\tP

## 6. GDELT archive-level audit (all downloaded files)
This scans **ZIP metadata** without decompressing all events. It identifies unreadable ZIP files, CRC failures only if explicitly tested (expensive), and declared uncompressed sizes. `testzip()` is optional and reads every ZIP member. A complete *record-level* GDELT audit remains required later; do not claim these metadata checks establish event-row quality.

**Run the CRC check only when you have sufficient time.**

In [10]:
# Task 1: GDELT Archive Validation

RUN_FULL_ZIP_CRC = True

archive_rows = []

for p in gdelt_files:
    row = {
        'filename': p.name,
        'compressed_bytes': p.stat().st_size,
        'zip_valid': None,
        'members': None,
        'uncompressed_bytes': None,
        'crc_error_member': None,
        'error': None
    }

    try:
        if zipfile.is_zipfile(p):
            with zipfile.ZipFile(p) as z:
                infos = [x for x in z.infolist() if not x.is_dir()]

                row.update(
                    zip_valid=True,
                    members=len(infos),
                    uncompressed_bytes=sum(x.file_size for x in infos)
                )

                if RUN_FULL_ZIP_CRC:
                    row['crc_error_member'] = z.testzip()

                    if row['crc_error_member'] is not None:
                        row['zip_valid'] = False

        else:
            row['zip_valid'] = False
            row['error'] = 'Not a valid ZIP archive'

    except Exception as e:
        row['zip_valid'] = False
        row['error'] = str(e)

    archive_rows.append(row)

archive_df = pd.DataFrame(archive_rows)

display(archive_df)

archive_df.to_csv(
    OUTPUT_DIR / 'gdelt_archive_inventory.csv',
    index=False
)

print("GDELT archive validation done")

,filename,compressed_bytes,zip_valid,members,uncompressed_bytes,crc_error_member,error
0,20171227.export.CSV.zip,8918869,True,1,51757873,None,None
1,20171228.export.CSV.zip,9439595,True,1,54143329,None,None
2,20171229.export.CSV.zip,8892460,True,1,51075034,None,None


GDELT archive validation done


## 7. Data dictionary template and source relationships
Fill in the actual meanings, units, identifiers, date semantics, and source version from your datasets' documentation. Do **not** assume GDELT event-country codes or event dates directly identify a stock; explain and justify any eventual joining/aggregation logic. GDELT covers only part of the price period.

A data dictionary is required even if you only use a subset of columns in later tasks.

In [12]:
# CREATE DATA DICTIONARY

# Define known stock price column information
price_info = {
    'date': ('Date', 'Trading date', 'YYYY-MM-DD', 'Trading day', 'NO'),
    'open': ('Float', 'Opening stock price', 'Price units (verify currency)', 'Trading day', 'NO'),
    'high': ('Float', 'Highest stock price during the day', 'Price units (verify currency)', 'Trading day', 'NO'),
    'low': ('Float', 'Lowest stock price during the day', 'Price units (verify currency)', 'Trading day', 'NO'),
    'close': ('Float', 'Closing stock price', 'Price units (verify currency)', 'Trading day', 'NO'),
    'volume': ('Integer', 'Number of shares traded', 'Shares', 'Trading day', 'NO'),
    'Name': ('String', 'Stock ticker symbol', 'N/A', 'Stock identifier', 'YES')
}

dictionary_rows = []

# Read stock price column names
if PRICE_FILE.exists():

    for name in pd.read_csv(PRICE_FILE, nrows=0).columns:

        info = price_info.get(
            name,
            ('VERIFY', 'FILL IN', 'FILL IN', 'FILL IN', 'YES/NO')
        )

        dictionary_rows.append({
            'source': 'prices',
            'field': name,
            'type': info[0],
            'meaning': info[1],
            'units': info[2],
            'time_semantics': info[3],
            'native_identifier': info[4]
        })

# Create DataFrame
dictionary_df = pd.DataFrame(
    dictionary_rows,
    columns=[
        'source',
        'field',
        'type',
        'meaning',
        'units',
        'time_semantics',
        'native_identifier'
    ]
)

# Display table
display(dictionary_df)

# Save as CSV
dictionary_df.to_csv(
    OUTPUT_DIR / 'data_dictionary_TEMPLATE.csv',
    index=False
)

print("Stock price data dictionary created successfully!")
print("Saved to: task1_outputs/data_dictionary_TEMPLATE.csv")
print("NOTE: GDELT fields still need to be added.")

,source,field,type,meaning,units,time_semantics,native_identifier
0,prices,date,Date,Trading date,YYYY-MM-DD,Trading day,NO
1,prices,open,Float,Opening stock price,Price units (verify currency),Trading day,NO
2,prices,high,Float,Highest stock price during the day,Price units (verify currency),Trading day,NO
3,prices,low,Float,Lowest stock price during the day,Price units (verify currency),Trading day,NO
4,prices,close,Float,Closing stock price,Price units (verify currency),Trading day,NO
5,prices,volume,Integer,Number of shares traded,Shares,Trading day,NO
6,prices,Name,String,Stock ticker symbol,N/A,Stock identifier,YES


Stock price data dictionary created successfully!
Saved to: task1_outputs/data_dictionary_TEMPLATE.csv
NOTE: GDELT fields still need to be added.


### 7.1 GDELT Data Dictionary and Schema Validation

This section identifies and documents the structure of the GDELT Event Database used in our project.

The GDELT dataset contains 58 columns, including event identifiers, dates, actors, event classifications, geographic information and source URLs.

To ensure that the downloaded files follow the expected structure, we:

1. Read a sample record directly from each GDELT ZIP archive without extracting the files.
2. Compare the number of columns against the expected 58-column GDELT event schema.
3. Create a data dictionary describing the fields, data types, meanings and time-related information.
4. Combine the GDELT data dictionary with the stock price data dictionary.

The combined data dictionary is exported as `data_dictionary_combined.csv` for documentation and future processing.

**Note:** This validation checks the column count of the first record in each archive. It does not verify every record or assess missing values and duplicates. Fields marked `VERIFY` or `FILL IN` still require further documentation.

In [13]:
# ==========================================
# TASK 1: GDELT DATA DICTIONARY
# ==========================================

import pandas as pd
import zipfile

# GDELT Event Database column names (58 columns)
gdelt_columns = [
    "GlobalEventID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_Lat", "Actor1Geo_Long",
    "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_Lat", "Actor2Geo_Long",
    "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_Lat", "ActionGeo_Long",
    "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

# Verify the actual number of columns in your GDELT files
gdelt_rows = []

for path in gdelt_files:

    with zipfile.ZipFile(path) as z:
        member = next(
            x for x in z.infolist() if not x.is_dir()
        )

        with z.open(member) as f:
            first_line = f.readline().decode(
                "utf-8", errors="replace"
            ).rstrip("\r\n")

            actual_columns = len(first_line.split("\t"))

            gdelt_rows.append({
                "filename": path.name,
                "actual_columns": actual_columns,
                "expected_columns": len(gdelt_columns),
                "schema_match": actual_columns == len(gdelt_columns)
            })

schema_df = pd.DataFrame(gdelt_rows)

display(schema_df)

# Only create the dictionary if all inspected files match
if schema_df.empty or not schema_df["schema_match"].all():
    print("STOP: GDELT schema mismatch. Check the files before proceeding.")

else:
    # Field descriptions for commonly used GDELT columns
    gdelt_info = {
        "GlobalEventID": ("String", "Unique GDELT event identifier", "N/A", "Event identifier", "YES"),
        "SQLDATE": ("Date", "Date associated with the event", "YYYYMMDD", "Event date", "NO"),
        "EventCode": ("String", "CAMEO event classification code", "N/A", "Event classification", "NO"),
        "GoldsteinScale": ("Float", "Goldstein score associated with the event type", "Score", "Event attribute", "NO"),
        "NumMentions": ("Integer", "Number of event mentions", "Count", "Event coverage", "NO"),
        "NumSources": ("Integer", "Number of distinct sources reporting the event", "Count", "Event coverage", "NO"),
        "NumArticles": ("Integer", "Number of articles mentioning the event", "Count", "Event coverage", "NO"),
        "AvgTone": ("Float", "Average tone of event coverage", "Tone score", "Event coverage", "NO"),
        "DATEADDED": ("Date", "Date event was added to GDELT", "YYYYMMDD", "Availability date", "NO"),
        "SOURCEURL": ("String", "URL of the source article", "URL", "Source reference", "NO")
    }

    gdelt_dictionary_rows = []

    for column in gdelt_columns:

        info = gdelt_info.get(
            column,
            ("VERIFY", "FILL IN", "FILL IN", "FILL IN", "YES/NO")
        )

        gdelt_dictionary_rows.append({
            "source": "GDELT",
            "field": column,
            "type": info[0],
            "meaning": info[1],
            "units": info[2],
            "time_semantics": info[3],
            "native_identifier": info[4]
        })

    gdelt_dictionary_df = pd.DataFrame(gdelt_dictionary_rows)

    # Combine with the stock price dictionary
    complete_dictionary_df = pd.concat(
        [dictionary_df, gdelt_dictionary_df],
        ignore_index=True
    )

    display(complete_dictionary_df)

    # Save combined dictionary
    complete_dictionary_df.to_csv(
        OUTPUT_DIR / "data_dictionary_combined.csv",
        index=False
    )

    print("Combined data dictionary created successfully!")
    print("Total fields:", len(complete_dictionary_df))
    print("Saved to: task1_outputs/data_dictionary_combined.csv")
    print("NOTE: Fields marked VERIFY/FILL IN still require documentation.")

,filename,actual_columns,expected_columns,schema_match
0,20171227.export.CSV.zip,58,58,True
1,20171228.export.CSV.zip,58,58,True
2,20171229.export.CSV.zip,58,58,True


,source,field,type,meaning,units,time_semantics,native_identifier
0,prices,date,Date,Trading date,YYYY-MM-DD,Trading day,NO
1,prices,open,Float,Opening stock price,Price units (verify currency),Trading day,NO
2,prices,high,Float,Highest stock price during the day,Price units (verify currency),Trading day,NO
3,prices,low,Float,Lowest stock price during the day,Price units (verify currency),Trading day,NO
4,prices,close,Float,Closing stock price,Price units (verify currency),Trading day,NO
...,...,...,...,...,...,...,...
60,GDELT,ActionGeo_Lat,VERIFY,FILL IN,FILL IN,FILL IN,YES/NO
61,GDELT,ActionGeo_Long,VERIFY,FILL IN,FILL IN,FILL IN,YES/NO
62,GDELT,ActionGeo_FeatureID,VERIFY,FILL IN,FILL IN,FILL IN,YES/NO
63,GDELT,DATEADDED,Date,Date event was added to GDELT,YYYYMMDD,Availability date,NO


Combined data dictionary created successfully!
Total fields: 65
Saved to: task1_outputs/data_dictionary_combined.csv
NOTE: Fields marked VERIFY/FILL IN still require documentation.


### 7.2 GDELT Data Quality Assessment

This section evaluates the quality of the downloaded GDELT event records by checking the total number of records, missing values, duplicate event identifiers, invalid dates, and data coverage.

These checks help identify potential data quality issues before the datasets are processed and used for further analysis.

In [14]:
# GDELT Data Quality Assessment

import pandas as pd
import zipfile
import io
from pathlib import Path

# Ensure output folder exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Store results
quality_results = []

# Track event IDs across all files
seen_event_ids = set()

# Expected number of GDELT columns
expected_columns = len(gdelt_columns)

# Important GDELT column positions (zero-based)
event_id_index = gdelt_columns.index("GlobalEventID")
event_date_index = gdelt_columns.index("SQLDATE")
date_added_index = gdelt_columns.index("DATEADDED")
event_code_index = gdelt_columns.index("EventCode")
tone_index = gdelt_columns.index("AvgTone")


# Process each GDELT ZIP file

for zip_path in gdelt_files:

    print(f"\nProcessing: {zip_path.name}")

    total_records = 0
    missing_values = 0
    duplicate_events = 0
    invalid_dates = 0
    invalid_column_counts = 0

    # Track missing values by field
    missing_by_field = {
        "GlobalEventID": 0,
        "SQLDATE": 0,
        "DATEADDED": 0,
        "EventCode": 0,
        "AvgTone": 0
    }

    # Track valid event dates
    valid_event_dates = []

    # Open ZIP file
    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.infolist():

            if member.is_dir():
                continue

            with z.open(member) as raw:

                text_file = io.TextIOWrapper(
                    raw,
                    encoding="utf-8",
                    errors="replace"
                )

                # Read every record
                for line in text_file:

                    total_records += 1

                    columns = line.rstrip("\r\n").split("\t")

                    # Check column count
                    if len(columns) != expected_columns:
                        invalid_column_counts += 1
                        continue

                    # Extract important fields
                    event_id = columns[event_id_index].strip()
                    event_date = columns[event_date_index].strip()
                    date_added = columns[date_added_index].strip()
                    event_code = columns[event_code_index].strip()
                    avg_tone = columns[tone_index].strip()

                    important_fields = {
                        "GlobalEventID": event_id,
                        "SQLDATE": event_date,
                        "DATEADDED": date_added,
                        "EventCode": event_code,
                        "AvgTone": avg_tone
                    }

                    # Check missing values

                    for field, value in important_fields.items():

                        if value == "":
                            missing_values += 1
                            missing_by_field[field] += 1

                    # Check duplicate event IDs

                    if event_id:

                        if event_id in seen_event_ids:
                            duplicate_events += 1

                        else:
                            seen_event_ids.add(event_id)

                    # Check invalid event dates

                    parsed_date = pd.to_datetime(
                        event_date,
                        format="%Y%m%d",
                        errors="coerce"
                    )

                    if pd.isna(parsed_date):
                        invalid_dates += 1

                    else:
                        valid_event_dates.append(parsed_date)

    # Calculate date coverage

    if valid_event_dates:

        earliest_date = min(valid_event_dates).strftime("%Y-%m-%d")
        latest_date = max(valid_event_dates).strftime("%Y-%m-%d")

    else:

        earliest_date = None
        latest_date = None

    # Store results

    quality_results.append({
        "filename": zip_path.name,
        "total_records": total_records,
        "missing_values": missing_values,
        "duplicate_event_ids": duplicate_events,
        "invalid_event_dates": invalid_dates,
        "invalid_column_counts": invalid_column_counts,
        "earliest_event_date": earliest_date,
        "latest_event_date": latest_date,
        "missing_GlobalEventID": missing_by_field["GlobalEventID"],
        "missing_SQLDATE": missing_by_field["SQLDATE"],
        "missing_DATEADDED": missing_by_field["DATEADDED"],
        "missing_EventCode": missing_by_field["EventCode"],
        "missing_AvgTone": missing_by_field["AvgTone"]
    })

    print(f"Total records: {total_records}")
    print(f"Missing values: {missing_values}")
    print(f"Duplicate event IDs: {duplicate_events}")
    print(f"Invalid dates: {invalid_dates}")
    print(f"Invalid column counts: {invalid_column_counts}")
    print(f"Date coverage: {earliest_date} to {latest_date}")


# Create Data Quality Summary

gdelt_quality_df = pd.DataFrame(quality_results)

print("\nGDELT DATA QUALITY SUMMARY")

display(gdelt_quality_df)


# Save Results to CSV

output_file = OUTPUT_DIR / "gdelt_data_quality.csv"

gdelt_quality_df.to_csv(
    output_file,
    index=False
)

print("\nData quality assessment completed!")
print(f"Results saved to: {output_file}")


Processing: 20171227.export.CSV.zip
Total records: 134988
Missing values: 0
Duplicate event IDs: 0
Invalid dates: 0
Invalid column counts: 0
Date coverage: 2007-12-30 to 2017-12-27

Processing: 20171228.export.CSV.zip
Total records: 141392
Missing values: 0
Duplicate event IDs: 0
Invalid dates: 0
Invalid column counts: 0
Date coverage: 2007-12-31 to 2017-12-28

Processing: 20171229.export.CSV.zip
Total records: 133245
Missing values: 0
Duplicate event IDs: 0
Invalid dates: 0
Invalid column counts: 0
Date coverage: 2008-01-01 to 2017-12-29

GDELT DATA QUALITY SUMMARY


,filename,total_records,missing_values,duplicate_event_ids,invalid_event_dates,invalid_column_counts,earliest_event_date,latest_event_date,missing_GlobalEventID,missing_SQLDATE,missing_DATEADDED,missing_EventCode,missing_AvgTone
0,20171227.export.CSV.zip,134988,0,0,0,0,2007-12-30,2017-12-27,0,0,0,0,0
1,20171228.export.CSV.zip,141392,0,0,0,0,2007-12-31,2017-12-28,0,0,0,0,0
2,20171229.export.CSV.zip,133245,0,0,0,0,2008-01-01,2017-12-29,0,0,0,0,0



Data quality assessment completed!
Results saved to: /Users/wileen/Desktop/y2sit/cloud/bigdataAssignment/task1_outputs/gdelt_data_quality.csv


## 8. Export local audit findings and limitations
Document which checks are complete and which remain provisional. Your final report must distinguish reference figures from observed figures and explain missing records, duplicates, corruption, coverage and acquisition limitations.

In [15]:
# Export Local Audit Findings and Limitations

summary = {
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'run_mode': RUN_MODE,
    'sources': SOURCE_REFERENCES,
    'acquired_utc': ACQUIRED_UTC,

    'price_file_count': len(price_files),
    'gdelt_file_count': len(gdelt_files),

    # Stock price quality assessment
    'price_quality': price_quality,

    # GDELT quality assessment
    'gdelt_archive_metadata_only': False,
    'gdelt_quality': gdelt_quality_df.to_dict(orient='records'),

    # Overall GDELT statistics
    'gdelt_total_records': int(gdelt_quality_df['total_records'].sum()),
    'gdelt_total_missing_values': int(gdelt_quality_df['missing_values'].sum()),
    'gdelt_total_duplicate_events': int(gdelt_quality_df['duplicate_event_ids'].sum()),
    'gdelt_total_invalid_dates': int(gdelt_quality_df['invalid_event_dates'].sum()),
    'gdelt_total_invalid_column_counts': int(gdelt_quality_df['invalid_column_counts'].sum()),

    # Limitations
    'limitations': [
        'Only three GDELT archives (27-29 December 2017) were assessed as a test sample.',
        'GDELT missing-value checks covered five selected fields, not all 58 columns.',
        'SQLDATE was validated, but DATEADDED and numeric value ranges require further validation.',
        'Source links, acquisition timestamps and remaining data dictionary descriptions require verification.',
        'Local inspection does not satisfy required EMR/HDFS execution evidence.',
        'Additional GDELT historical data is required for the final stock prediction analysis.'
    ]
}

# Save summary as JSON
output_file = OUTPUT_DIR / 'task1_local_summary.json'

output_file.write_text(
    json.dumps(summary, indent=2),
    encoding='utf-8'
)

print('Local audit summary exported successfully!')
print('Total GDELT records:', summary['gdelt_total_records'])
print('Total missing values:', summary['gdelt_total_missing_values'])
print('Total duplicate events:', summary['gdelt_total_duplicate_events'])
print('Total invalid dates:', summary['gdelt_total_invalid_dates'])
print('Outputs saved in:', OUTPUT_DIR)

Local audit summary exported successfully!
Total GDELT records: 409625
Total missing values: 0
Total duplicate events: 0
Total invalid dates: 0
Outputs saved in: /Users/wileen/Desktop/y2sit/cloud/bigdataAssignment/task1_outputs


## 9. EMR-only: stage original files in HDFS
**Where:** Jupyter notebook connected to the EMR primary node (or EMR terminal). **Do not run locally.**

**Before this step:** Member 5 must create the prescribed cluster (one primary + two core m5.xlarge nodes, no task nodes/autoscaling), install/configure required applications, and transfer original files to paths accessible from the primary node. Merely uploading the notebook does **not** upload its input files. You may stage inputs through S3 if available; HDFS is required for the assessed corpus.

The following cell copies files from the notebook's accessible filesystem into HDFS, preserves originals, and fails if a command fails. Large GDELT uploads may take substantial time and storage. Run only with the approved workload profile. **The `emr` flag is a safeguard, not proof that a real EMR connection exists.**

In [ ]:
def run_checked(args):
    print('$', ' '.join(map(str,args)))
    completed = subprocess.run([str(x) for x in args], text=True, capture_output=True)
    if completed.stdout: print(completed.stdout[:5000])
    if completed.stderr: print(completed.stderr[:3000])
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed ({completed.returncode}): {args}')
    return completed.stdout

if RUN_MODE == 'emr':
    if not price_files or not gdelt_files:
        raise RuntimeError('Missing staged input files. Set paths in Section 1.')
    run_checked(['hdfs','dfs','-mkdir','-p',f'{HDFS_BASE}/prices',f'{HDFS_BASE}/gdelt'])
    for p in price_files:
        run_checked(['hdfs','dfs','-put','-f',str(p),f'{HDFS_BASE}/prices/{p.name}'])
    for p in gdelt_files:
        run_checked(['hdfs','dfs','-put','-f',str(p),f'{HDFS_BASE}/gdelt/{p.name}'])
    print('HDFS upload complete')
else:
    print('Skipped: RUN_MODE is local. EMR execution required for assessment.')

## 10. EMR-only: verify HDFS corpus and record evidence
HDFS `-ls` and `-du` show actual stored files and bytes. Compare uploaded files with the inventory. **HDFS checksums are not automatically the same as SHA-256**; to verify exact bytes, stream each HDFS file through SHA-256 and compare with the original manifest (potentially slow for GDELT). Keep command outputs, cluster ID, timestamps, configurations, and job/query IDs for the final report. HDFS content is lost when the EMR cluster is terminated.

In [ ]:
if RUN_MODE == 'emr':
    listings = {}
    for src in ('prices','gdelt'):
        listings[src] = run_checked(['hdfs','dfs','-ls','-R',f'{HDFS_BASE}/{src}'])
        run_checked(['hdfs','dfs','-du','-s',f'{HDFS_BASE}/{src}'])
    (OUTPUT_DIR/'hdfs_listing.txt').write_text('\n'.join(listings.values()), encoding='utf-8')
    print('HDFS evidence written. Save it outside the cluster before termination.')
else:
    print('Skipped: no EMR/HDFS run performed.')

## 11. EMR-only: optional exact-file verification
This verifies each HDFS file against its **original SHA-256**, streaming the bytes rather than downloading entire files to memory. It can be slow for large GDELT inputs. Run it only when the files have been uploaded and the manifest is complete.

In [ ]:
RUN_HDFS_SHA256 = False  # change to True on EMR when ready
if RUN_MODE == 'emr' and RUN_HDFS_SHA256:
    verification = []
    for row in inventory:
        dest = f"{HDFS_BASE}/{row['source']}/{row['filename']}"
        digest = hashlib.sha256()
        proc = subprocess.Popen(['hdfs','dfs','-cat',dest], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for block in iter(lambda: proc.stdout.read(8*1024*1024), b''):
            digest.update(block)
        err = proc.stderr.read().decode(errors='replace')
        rc = proc.wait()
        if rc != 0: raise RuntimeError(f'HDFS read failed for {dest}: {err}')
        verification.append({'hdfs_path':dest, 'local_sha256':row['sha256'],
                             'hdfs_sha256':digest.hexdigest(), 'match':digest.hexdigest()==row['sha256']})
    verification_df = pd.DataFrame(verification)
    display(verification_df)
    verification_df.to_csv(OUTPUT_DIR/'hdfs_sha256_verification.csv', index=False)
else:
    print('Exact HDFS SHA-256 verification not run.')

## 12. Reproducibility checklist (fill before submission)
- [ ] Source links, versions, acquisition dates, filenames and SHA-256 recorded.
- [ ] Price dataset date coverage, identifiers, schemas, units and row counts measured.
- [ ] GDELT archive coverage, file counts and **full-workload** record-quality results measured or clearly identified as pending.
- [ ] Missing, duplicate, invalid and rejected/retained data handling documented (actual cleaning belongs to Task 2).
- [ ] Original source files preserved in HDFS on EMR with evidence.
- [ ] HDFS directory structure and configuration recorded.
- [ ] Notebook, helper scripts, and logs saved outside temporary EMR/HDFS storage.
- [ ] EMR setup and fresh-session rerun steps written for Appendix A.
- [ ] Source relationship/availability timing assumptions documented before integrating GDELT with price data.
- [ ] No credentials or raw datasets in the final submission ZIP.

**Handoff to Member 2:** agree on paths, actual column names, timestamp semantics, source coverage, quality issues and the intended cleaning/rejection rules. **Handoff to Member 5:** provide staged data references, manifest, notebook, configuration and evidence requirements.

**Note:** the PDF inconsistently labels the team-contributions appendix (page 6: D; page 9: C); confirm with the teaching team.